# 6.3 性能测量

第3、4章已经用`msprof`比较过不同实现的耗时。本节进一步解释采集命令和报告字段，并介绍在Python中调用`do_bench`取得耗时的方法，帮助你独立测量自己的算子。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">你想得到什么</th>
<th style="text-align: left; vertical-align: top;">使用方式</th>
<th style="text-align: left; vertical-align: top;">结果在哪里</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">在程序中取得一个耗时数值，方便打印或记录</td>
<td style="text-align: left; vertical-align: top;"><code>do_bench(backend="msprof")</code></td>
<td style="text-align: left; vertical-align: top;">函数返回值，单位是毫秒</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">保留每次调用的记录，查看单次耗时或自行统计</td>
<td style="text-align: left; vertical-align: top;"><code>msprof</code>命令行</td>
<td style="text-align: left; vertical-align: top;">输出目录中的CSV报告</td>
</tr>
</tbody>
</table>

两种方式都使用CANN的性能采集能力。`do_bench`是TileLang提供的Python接口，通过`torch_npu`调用采集接口；`msprof`是命令行工具，用来启动程序、采集性能数据并生成报告。

下面用同一个GEMM示范，分别查看**函数返回的耗时数值**和**报告中的调用记录**。

<p class="course-note"><small>线上环境没有昇腾950，可直接阅读页内代码和实测结果；自行运行请按1.4准备环境。本例使用CANN 9.1.0、PyTorch/torch_npu 2.11.0。</small></p>

<p class="course-note"><small>在Notebook中运行时，以本章目录为工作目录，从下方准备单元开始顺序执行。完整程序和运行辅助代码可展开查看；源码保存单元执行后，再运行对应程序。</small></p>

In [1]:
from pathlib import Path
import re
import subprocess
import sys

Path("src").mkdir(exist_ok=True)
Path("answer").mkdir(exist_ok=True)

def run_and_show(command):
    # NPU程序在独立进程中运行；当前Notebook内核不导入torch_npu。
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    # 只省略安装目录；数值、校验结果和报错内容保持原样。
    output = result.stdout.replace(str(Path.cwd()), ".")
    output = re.sub(r"/home/[^\s]+", "<已省略安装路径>", output)
    print(output, end="")
    result.check_returncode()
    return result


## 1. 准备一个可以调用的kernel

本例计算 **`C = A @ B.T`**，采用以下固定配置：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">项目</th>
<th style="text-align: left; vertical-align: top;">本例设置</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">矩阵尺寸</td>
<td style="text-align: left; vertical-align: top;"><code>M = N = K = 512</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">数据类型</td>
<td style="text-align: left; vertical-align: top;">输入为float16，累加和输出为float32</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">分块与流水</td>
<td style="text-align: left; vertical-align: top;"><code>BM = BN = BK = 64</code>，<code>num_stages=2</code></td>
</tr>
</tbody>
</table>

本例每个逻辑任务只负责一个输出块。`T.Pipelined(K // BK, num_stages=2)`的每轮处理一段K，计算结果继续累加到同一个`acc`；4.3的流水循环则每轮处理一个新的输出块。**两处循环处理的对象不同。** 本节固定上述配置，重点学习Host侧如何测量耗时。

假设`lesson_gemm`是已经写好的TileLang函数，先编译它、准备输入，并检查一次完整输出：

```python
kernel = tilelang.compile(lesson_gemm, target="ascend", out_idx=-1)
torch.manual_seed(20260916)
a_cpu = (torch.randn(M, K) * 0.125).half()
b_cpu = (torch.randn(N, K) * 0.125).half()
a_npu, b_npu = a_cpu.npu(), b_cpu.npu()

c_npu = kernel(a_npu, b_npu)
actual = c_npu.cpu()
expected = a_cpu.float() @ b_cpu.float().T
torch.testing.assert_close(actual, expected, rtol=0.002, atol=0.002)
```

`tilelang.compile(...)`返回一个可以从Python调用的函数，这里将它保存为`kernel`。`a_npu`和`b_npu`是已经放到NPU上的输入张量。

执行`c_npu = kernel(a_npu, b_npu)`时，程序会启动这个GEMM算子，并返回保存输出的NPU张量。下一行`c_npu.cpu()`将结果取回CPU，供后面的正确性检查使用。

本节要查看的是**这次GEMM在NPU上执行的耗时**，其中包含kernel内部的GM读写和计算。编译程序、准备输入和检查结果所花的时间，不是我们要读取的GEMM耗时。

下面的两种测量方式使用同一个GEMM，各自给出可以独立运行的完整程序。

## 2. 使用do_bench，直接取得耗时

### 2.1 导入并调用

`do_bench`由TileLang提供，从 `tilelang.profiler` 导入。

```python
from tilelang.profiler import do_bench

latency_ms = do_bench(
    lambda: kernel(a_npu, b_npu),
    backend="msprof",
    _n_warmup=30,
    _n_repeat=100,
    cache_size=256,
)
print(f"latency_ms = {latency_ms:.8f}")
print(f"latency_us = {latency_ms * 1000:.5f}")
```

`lambda: kernel(a_npu, b_npu)`创建了一个不带参数的小函数。`do_bench`每调用一次这个小函数，就会执行一次`kernel(a_npu, b_npu)`，也就是启动一次GEMM计算。预热、重复执行和耗时统计由`do_bench`完成。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数</th>
<th style="text-align: left; vertical-align: top;">本例中的作用</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>backend="msprof"</code></td>
<td style="text-align: left; vertical-align: top;">通过<code>torch_npu</code>调用CANN采集接口，从设备执行记录统计耗时；本节显式指定该后端</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>_n_warmup=30</code></td>
<td style="text-align: left; vertical-align: top;">正式采样前预热30次，使程序先运行起来</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>_n_repeat=100</code></td>
<td style="text-align: left; vertical-align: top;">正式采样100次，当前实现按重复次数计算平均耗时</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>cache_size=256</code></td>
<td style="text-align: left; vertical-align: top;">使用256 MiB缓冲区刷新数据缓存；各次正式采样前都执行相同的缓存处理</td>
</tr>
</tbody>
</table>

<p class="course-note"><small>调用次数：工具还会进行初始化和估时调用，所以整个程序的kernel调用次数不只是30+100。用于刷新缓存的kernel会从最终耗时中排除。</small></p>

**两种缓存设置：** `cache_size`是刷新**数据缓存**所用的缓冲区大小，不是设备L2容量；`tilelang.disable_cache()`关闭的是TileLang的**编译缓存**。

### 2.2 运行并读懂返回值

**测量前，确认设备没有其他计算任务争用。** 完整程序如下，已包含编译、输入准备、正确性检查和计时。

In [2]:
%%writefile src/gemm_bench.py
import tilelang
import tilelang.language as T
import torch
import torch_npu
from tilelang.profiler import do_bench

tilelang.disable_cache()

M = N = K = 512
BM = BN = BK = 64
WARMUP = 30
REPEAT = 100


@T.prim_func
def lesson_gemm(
    A: T.Tensor((M, K), "float16"),
    B: T.Tensor((N, K), "float16"),
    C: T.Tensor((M, N), "float32"),
):
    with T.Kernel((M // BM) * (N // BN)) as bx:
        row = bx // (N // BN)
        col = bx % (N // BN)
        a = T.alloc_l1((BM, BK), "float16")
        b = T.alloc_l1((BN, BK), "float16")
        acc = T.alloc_l0c((BM, BN), "float32")
        for ki in T.Pipelined(K // BK, num_stages=2):
            T.copy(A[row * BM:(row + 1) * BM,
                     ki * BK:(ki + 1) * BK], a)
            T.copy(B[col * BN:(col + 1) * BN,
                     ki * BK:(ki + 1) * BK], b)
            T.gemm(a, b, acc, transpose_B=True, clear_accum=(ki == 0))
        T.copy(acc, C[row * BM:(row + 1) * BM,
                      col * BN:(col + 1) * BN])


def main():
    torch.npu.set_device(0)
    kernel = tilelang.compile(lesson_gemm, target="ascend", out_idx=-1)
    torch.manual_seed(20260916)
    a_cpu = (torch.randn(M, K) * 0.125).half()
    b_cpu = (torch.randn(N, K) * 0.125).half()
    a_npu, b_npu = a_cpu.npu(), b_cpu.npu()

    actual = kernel(a_npu, b_npu).cpu()
    expected = a_cpu.float() @ b_cpu.float().T
    torch.testing.assert_close(actual, expected, rtol=0.002, atol=0.002)
    print("Verification passed!", flush=True)

    latency_ms = do_bench(
        lambda: kernel(a_npu, b_npu),
        backend="msprof",
        _n_warmup=WARMUP,
        _n_repeat=REPEAT,
        cache_size=256,
    )
    print(f"latency_ms = {latency_ms:.8f}")
    print(f"latency_us = {latency_ms * 1000:.5f}")


if __name__ == "__main__":
    main()


Overwriting src/gemm_bench.py


在终端中运行：

```bash
python src/gemm_bench.py
```

也可以执行下方Notebook单元。**先看输出末尾的 `latency_ms` 和 `latency_us`。**

In [3]:
run_result = run_and_show([sys.executable, "-u", 'src/gemm_bench.py'])


Loading tilelang libs from dev root: <已省略安装路径>
[W918 20:21:46.567559526 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 20:21:54  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `lesson_gemm` with `out_idx=-1`
2026-09-18 20:22:06  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `lesson_gemm`
Verification passed!
2026-09-18 20:22:06  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `_tilelang_profiler_cache_flush` with `out_idx=[]`
2026-09-18 20:22:18  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `_tilelang_profiler_cache_flush`
latency_ms = 0.00673199
latency_us = 6.73199


<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">输出项</th>
<th style="text-align: left; vertical-align: top;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><strong><code>latency_ms</code></strong></td>
<td style="text-align: left; vertical-align: top;">正式采样的平均设备耗时，单位为毫秒</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><strong><code>latency_us</code></strong></td>
<td style="text-align: left; vertical-align: top;">同一耗时换算成微秒，即 <code>latency_ms × 1000</code></td>
</tr>
</tbody>
</table>

`latency_ms`乘以1000即为`latency_us`。这里的耗时是100次正式采样的平均设备耗时，不是100次的总和，也不包含程序启动或编译过程。

本例交给`do_bench`的函数只启动一个GEMM kernel。测量自己的算子时，把这个调用换成自己的kernel，并在调用前完成编译、输入准备和正确性检查即可。如果被测函数会启动多个kernel，返回值对应这些kernel的耗时汇总，不能把它当成其中某一个kernel的耗时。

**保存结果：** 可以将返回值写入自己的表格或文件；需要保留逐次调用的CSV记录时，使用下一部分的 `msprof` 命令行方式。

<details>
<summary>补充：接口来源与记录保存</summary>

实现位于TileLang仓库的 `tilelang/profiler/bench.py`，TileKernels的Ascend测试也使用该接口。当前实现会解析原始记录，并在结束后清理临时采集目录。

</details>

<p class="course-note"><small>完整程序：<a href="src/gemm_bench.py">gemm_bench.py</a>。再次测量时，以新输出的数值为准。</small></p>

## 3. 使用msprof，采集并保存报告

### 3.1 准备要运行的程序

msprof可以直接采集一个正常运行的算子程序。本例使用[gemm_msprof.py](src/gemm_msprof.py)：它准备输入，调用一次GEMM，再检查输出是否正确。关键语句是：

```python
c_npu = kernel(a_npu, b_npu)
actual = c_npu.cpu()
expected = a_cpu.float() @ b_cpu.float().T
torch.testing.assert_close(actual, expected, rtol=0.002, atol=0.002)
```

第一行启动GEMM，第二行将计算结果取回CPU；后两行计算参考结果并检查完整输出。使用msprof运行这个程序后，就能在报告中找到这一次GEMM调用的记录。

完整程序如下，可展开查看。

In [4]:
%%writefile src/gemm_msprof.py
import tilelang
import tilelang.language as T
import torch
import torch_npu

tilelang.disable_cache()

M = N = K = 512
BM = BN = BK = 64


@T.prim_func
def lesson_gemm(
    A: T.Tensor((M, K), "float16"),
    B: T.Tensor((N, K), "float16"),
    C: T.Tensor((M, N), "float32"),
):
    with T.Kernel((M // BM) * (N // BN)) as bx:
        row = bx // (N // BN)
        col = bx % (N // BN)
        a = T.alloc_l1((BM, BK), "float16")
        b = T.alloc_l1((BN, BK), "float16")
        acc = T.alloc_l0c((BM, BN), "float32")
        for ki in T.Pipelined(K // BK, num_stages=2):
            T.copy(A[row * BM:(row + 1) * BM,
                     ki * BK:(ki + 1) * BK], a)
            T.copy(B[col * BN:(col + 1) * BN,
                     ki * BK:(ki + 1) * BK], b)
            T.gemm(a, b, acc, transpose_B=True, clear_accum=(ki == 0))
        T.copy(acc, C[row * BM:(row + 1) * BM,
                      col * BN:(col + 1) * BN])


def main():
    torch.npu.set_device(0)
    kernel = tilelang.compile(lesson_gemm, target="ascend", out_idx=-1)
    torch.manual_seed(20260916)
    a_cpu = (torch.randn(M, K) * 0.125).half()
    b_cpu = (torch.randn(N, K) * 0.125).half()
    a_npu, b_npu = a_cpu.npu(), b_cpu.npu()

    c_npu = kernel(a_npu, b_npu)
    actual = c_npu.cpu()
    expected = a_cpu.float() @ b_cpu.float().T
    torch.testing.assert_close(actual, expected, rtol=0.002, atol=0.002)
    print("Verification passed!", flush=True)



if __name__ == "__main__":
    main()


Overwriting src/gemm_msprof.py


### 3.2 执行采集命令

```bash
msprof --output=./prof_msprof --type=text --task-time=on \
    python src/gemm_msprof.py
```

从左到右读这条命令：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">部分</th>
<th style="text-align: left; vertical-align: top;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>msprof</code></td>
<td style="text-align: left; vertical-align: top;">启动性能采集工具</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>--output=./prof_msprof</code></td>
<td style="text-align: left; vertical-align: top;">将本次采集文件放在<code>prof_msprof</code>目录下</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>--type=text</code></td>
<td style="text-align: left; vertical-align: top;">导出便于查看的文本报告，包括后面使用的CSV</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>--task-time=on</code></td>
<td style="text-align: left; vertical-align: top;">开启任务耗时相关数据的采集</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>python src/gemm_msprof.py</code></td>
<td style="text-align: left; vertical-align: top;">被采集的普通Python程序</td>
</tr>
</tbody>
</table>

**等报告导出完成再读数据：** 程序打印完毕后，`msprof`还需要解析并导出报告。请等待工具完成。

<p class="course-note"><small>再次采集时，可使用<code>--output=./prof_msprof_run2</code>等新目录区分结果；下方Notebook单元会自动创建独立目录。</small></p>

In [5]:
import tempfile
Path('prof_msprof').mkdir(exist_ok=True)
profile_dir = Path(tempfile.mkdtemp(prefix="run_", dir='prof_msprof'))
profile_result = run_and_show([
    "msprof", f"--output={profile_dir}", "--type=text", "--task-time=on",
    sys.executable, "-u", 'src/gemm_msprof.py',
])


[INFO] Start profiling....
Loading tilelang libs from dev root: <已省略安装路径>
[W918 20:22:25.877799100 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 20:22:33  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `lesson_gemm` with `out_idx=-1`
2026-09-18 20:22:45  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `lesson_gemm`
Verification passed!
[INFO] Start export data in PROF_000001_20260918202232145_00942262QAAEQBME.
Fri 18 Sep 2026 20:22:50 [WARNING] [MSVP] [942682] msprof_entrance.py: Msprof analysis is parsing data as root, which may cause potential system security risks.
Fri 18 Sep 2026 20:22:50 [INFO] [MSVP] [942684] msprof_common.py: Start analyzing data in "./prof_msprof/run_uddkgc0b/PROF_000001_20260918202232145_00942262QAAEQBME/host" ...
Fri 18 Sep 2026 20:22:50 [INFO] [MSVP] [942684] msprof_common.py: It 

采集完成后，继续查看下方的CSV报告。本次完整采集日志保留在上方运行单元中，可按需展开。

<p class="course-note"><small>完整程序：<a href="src/gemm_msprof.py">gemm_msprof.py</a>。</small></p>

## 4. 查看报告，找到自己算子的耗时

### 4.1 先找到两份CSV

采集目录中的主要文件如下。

<p class="course-note"><small>文件名说明：目录名和文件名中的时间戳以实际结果为准，下面用省略号表示。</small></p>

```text
prof_msprof/
└── PROF_.../
    └── mindstudio_profiler_output/
        ├── op_summary_....csv
        ├── op_statistic_....csv
        ├── api_statistic_....csv
        ├── task_time_....csv
        └── msprof_....json
```

先打开`op_summary`和`op_statistic`即可。可以直接用表格软件打开，按列名找到相应数据。两份报告各回答一个问题：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">文件</th>
<th style="text-align: left; vertical-align: top;">怎样看</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>op_summary_*.csv</code></td>
<td style="text-align: left; vertical-align: top;">每条任务记录一行。按<code>Op Name</code>找到自己的kernel，读 <strong><code>Task Duration(us)</code></strong> 查看这次调用的耗时</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>op_statistic_*.csv</code></td>
<td style="text-align: left; vertical-align: top;">按算子类型等字段汇总。查看<code>Count</code>、<code>Total Time(us)</code>和<code>Avg Time(us)</code>，了解采集范围内的次数、总耗时和平均耗时</td>
</tr>
</tbody>
</table>

本例TileLang函数名是`lesson_gemm`，报告中的名称是`lesson_gemm_kernel`。测量自己的算子时，应先查看`Op Name`的实际内容，再按完整名称筛选，不要直接对整份CSV取平均。

**`Task Duration(us)`表示设备任务耗时**，包含任务在加速器侧的调度、执行及结束响应；它不是纯Cube计算时间。`Task Start Time(us)`是开始时刻，`Task Wait Time(us)`是任务间隔，都不是本次任务的耗时。

<p class="course-note"><small>参考资料：<a href="https://www.hiascend.com/document/detail/zh/mindstudio/830/T%26ITools/Profiling/atlasprofiling_16_0151.html">报告字段说明</a>。</small></p>

### 4.2 对照本例，读取单次耗时和汇总值

**先看明细中的 `Task Duration(us)`，再对照汇总中的 `Count`、`Total Time(us)` 和 `Avg Time(us)`。**

下面两张表由本次CSV按目标kernel名称筛选生成。读取代码可展开查看；重新采集后再次执行该单元，表格也会更新。

In [6]:
import csv
import shutil
from IPython.display import Markdown, display

def load_target(pattern, name_field):
    paths = list(profile_dir.rglob(pattern))
    if len(paths) != 1:
        raise RuntimeError(f"本次目录中应有一份{pattern}，实际找到{len(paths)}份")
    with paths[0].open(encoding="utf-8-sig", newline="") as file:
        rows = [row for row in csv.DictReader(file) if row[name_field] == 'lesson_gemm_kernel']
    if not rows:
        raise RuntimeError("本次报告中没有找到目标算子")
    return paths[0], rows

summary_path, summary_rows = load_target("op_summary_*.csv", "Op Name")
statistic_path, statistic_rows = load_target("op_statistic_*.csv", "OP Type")

def show_table(title, rows, columns):
    lines = ["**" + title + "**", "", "| " + " | ".join(columns) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    lines += ["| " + " | ".join(row[column].strip() for column in columns) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

print("本次明细报告：", summary_path.relative_to(Path.cwd()))
print("本次汇总报告：", statistic_path.relative_to(Path.cwd()))
show_table("单次调用", summary_rows, ['Op Name', 'Task ID', 'Task Duration(us)'])
show_table("汇总结果", statistic_rows, ['OP Type', 'Count', 'Total Time(us)', 'Avg Time(us)'])

# 保存同一批报告，便于离线打开；表格内容始终取自上面的本次采集目录。
sample_dir = Path('data/msprof_sample')
sample_dir.mkdir(parents=True, exist_ok=True)
shutil.copyfile(summary_path, sample_dir / "op_summary_current.csv")
_ = shutil.copyfile(statistic_path, sample_dir / "op_statistic_current.csv")


本次明细报告： prof_msprof/run_uddkgc0b/PROF_000001_20260918202232145_00942262QAAEQBME/mindstudio_profiler_output/op_summary_20260918202254.csv
本次汇总报告： prof_msprof/run_uddkgc0b/PROF_000001_20260918202232145_00942262QAAEQBME/mindstudio_profiler_output/op_statistic_20260918202254.csv


**单次调用**

| Op Name | Task ID | Task Duration(us) |
| --- | --- | --- |
| lesson_gemm_kernel | 4 | 9.754 |

**汇总结果**

| OP Type | Count | Total Time(us) | Avg Time(us) |
| --- | --- | --- | --- |
| lesson_gemm_kernel | 1 | 9.754 | 9.754 |

`Count`是调用次数，`Total Time(us)`是这些调用的耗时总和，`Avg Time(us)`是平均耗时。本例只调用了一次GEMM，所以次数为1，总耗时和平均耗时与明细中的这次耗时一致。

**注意测量条件：** do_bench给出预热后的多次采样平均值，本例msprof记录未额外预热的单次调用，两者不宜直接比较。

如果自己的程序调用了多次同一个算子，明细表会列出各次调用，汇总表则给出相应的次数和统计值。汇总范围包含程序实际执行的全部调用；如果程序中有校验或预热调用，它们也会计入。

<p class="course-note"><small>相关文件：<a href="data/msprof_sample/op_summary_current.csv">调用明细CSV</a> · <a href="data/msprof_sample/op_statistic_current.csv">汇总CSV</a>。表格与文件来自同一批采集记录。</small></p>

## 5. 换成自己的算子

使用`do_bench`时，保留第2部分的测量代码，把`lambda`中的调用替换成自己的kernel及输入；运行后读取返回值，注意它的单位是毫秒。

使用`msprof`时，将采集命令末尾换成自己的Python程序及其参数。报告生成后，先确认kernel名称，再查看`Task Duration(us)`；需要平均值时，明确是否包含校验和预热记录。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">遇到的情况</th>
<th style="text-align: left; vertical-align: top;">检查位置</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>msprof</code>命令找不到</td>
<td style="text-align: left; vertical-align: top;">检查是否已按1.4初始化CANN环境；用<code>msprof --help</code>确认工具可用</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">没找到CSV，或读到了上一次的结果</td>
<td style="text-align: left; vertical-align: top;">确认本次程序与报告导出已完成，并选中本次输出目录下的<code>PROF_...</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">筛选后没有目标记录</td>
<td style="text-align: left; vertical-align: top;">查看<code>Op Name</code>列中的实际名称，再按名称筛选</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">记录数不是预期值</td>
<td style="text-align: left; vertical-align: top;">核对程序实际调用了几次目标算子，并确认打开的是本次报告</td>
</tr>
</tbody>
</table>

**记录结果时一并保存：** 输入形状、数据类型、测量工具、单位，以及预热和缓存设置，便于以后按同样条件重新测量。

<details>
<summary>检查自己是否读懂了结果</summary>

1. `do_bench`的返回值以毫秒为单位，乘以1000得到微秒。
2. 本例`msprof`报告中的`Count=1`表示目标算子调用了一次，因此总耗时和平均耗时相同。
3. 本例要看的耗时列是`Task Duration(us)`，不是`Task Start Time(us)`或`Task Wait Time(us)`。

</details>

[章节导航](README.md) · [上一节](06.02_debugging.ipynb) · [下一节](06.04_capstone.ipynb)